# Seed variability

Trains the selected configuration (YOLOv26l, online augmentation, no offline
augmentation) with one random seed, keeping the data partition fixed.
Deterministic mode is disabled on purpose, so that the non-deterministic GPU
operations also contribute to the measured variation. Repeat with seeds 0 to
9, one per session, and run the summary cell when all seeds are done.

## Setup

Select a GPU runtime (**Runtime > Change runtime type**). Outputs are written
to Google Drive, so that a run interrupted by the end of the session can be
resumed. The Roboflow API key is read from the Colab secret
`ROBOFLOW_API_KEY` (key icon in the left panel).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics==8.4.157 roboflow

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path
from types import SimpleNamespace
import statistics

from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/grape-segmentation"
os.makedirs(OUTPUT, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

## Configuration

Hyperparameters kept identical across every run.

In [ ]:
IMAGE_SIZE = 1024

MAX_DETECTIONS = 300

# Hyperparameters kept identical across every run. Most are Ultralytics
# defaults, written out so that a different library version cannot change
# them silently.
TRAINING = dict(
    imgsz=IMAGE_SIZE,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    # Masks of overlapping instances are kept separate; with the default
    # (True) a berry mask overwrites the mask of the bunch that contains it.
    overlap_mask=False,
    mask_ratio=4,
    close_mosaic=10,
    amp=True,
    max_det=MAX_DETECTIONS,
    cache="disk",
    plots=True,
    val=True,
)

ONLINE_AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    erasing=0.4,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
)

NO_ONLINE_AUGMENTATION = {name: 0.0 for name in ONLINE_AUGMENTATION}

NO_ONLINE_AUGMENTATION["close_mosaic"] = 0

# Epoch limit and early-stopping patience. Runs on the dataset with offline
# augmentation see ten copies of each image per epoch, so they use fewer epochs.
SCHEDULE_OFFLINE = dict(epochs=150, patience=40)

SCHEDULE_NO_OFFLINE = dict(epochs=600, patience=80)

SCHEDULE_VARIABILITY = dict(epochs=600, patience=120)

## Dataset

The Roboflow export stores part of the instances as bounding boxes. They are
converted into four-vertex polygons, because Ultralytics discards label files
that mix boxes and polygons in segmentation training. Version 14 is the export
without offline augmentation.

In [ ]:
SPLITS = ("train", "valid", "test")


def download_dataset(workspace, project, version, location):
    """Download a Roboflow dataset version in YOLO segmentation format.

    The API key is read from the ROBOFLOW_API_KEY environment variable.
    Returns the path of the generated data.yaml.
    """
    from roboflow import Roboflow

    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("set the ROBOFLOW_API_KEY environment variable")
    rf = Roboflow(api_key=api_key)
    rf.workspace(workspace).project(project).version(version).download("yolov11", location=str(location))
    return Path(location) / "data.yaml"


def _parse(line):
    parts = line.split()
    if not parts:
        return None, None, None
    values = [float(v) for v in parts[1:]]
    if len(values) == 4:
        return parts[0], values, "box"
    if len(values) >= 6 and len(values) % 2 == 0:
        return parts[0], values, "polygon"
    return parts[0], values, None


def _box_to_polygon(values):
    xc, yc, w, h = values
    x1, x2 = xc - w / 2, xc + w / 2
    y1, y2 = yc - h / 2, yc + h / 2
    return [min(max(p, 0.0), 1.0) for p in (x1, y1, x2, y1, x2, y2, x1, y2)]


def convert_boxes_to_polygons(root):
    """Rewrite bounding-box label lines as four-vertex polygons.

    The Roboflow export stores part of the instances as boxes. A label file
    that mixes boxes and polygons is discarded by Ultralytics in segmentation
    training, so every box is converted to the polygon of its corners.
    Returns the number of converted lines per class.
    """
    root = Path(root)
    converted = Counter()
    for split in SPLITS:
        folder = root / split / "labels"
        if not folder.is_dir():
            continue
        for path in folder.glob("*.txt"):
            lines, changed = [], False
            for line in path.read_text().splitlines():
                cls, values, kind = _parse(line)
                if kind == "box":
                    points = _box_to_polygon(values)
                    lines.append(cls + " " + " ".join(f"{p:.6f}" for p in points))
                    converted[cls] += 1
                    changed = True
                elif line.strip():
                    lines.append(line.strip())
            if changed:
                path.write_text("\n".join(lines) + "\n")
    # Stale label caches would keep the discarded files out of training.
    for cache in root.rglob("*.cache"):
        cache.unlink()
    return converted

In [ ]:
VERSION = 14  # @param {type:"integer"}

data_yaml = download_dataset("hugo-leal-cn7iw", "grapes-mic3h", VERSION, "/content/dataset")
converted = convert_boxes_to_polygons("/content/dataset")
print(f"data.yaml: {data_yaml}")
print(f"box labels converted to polygons: {dict(converted) or 'none'}")

## Metrics

Precision, recall, mAP@50 and mAP@50-95 reported by Ultralytics for boxes and
masks, and F1 computed from precision and recall.

In [ ]:
METRICS = ("P", "R", "mAP50", "mAP50_95")


def f1(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0


def metrics_dict(results, names):
    """Precision, recall, F1, mAP50 and mAP50-95 for boxes and masks.

    Per-class F1 is computed from the precision and recall reported by the
    library; F1 over both classes is the mean of the per-class values, the
    same criterion the library applies to precision and recall.
    """
    def block(m, i=None):
        if i is None:
            values = dict(P=m.mp, R=m.mr, mAP50=m.map50, mAP50_95=m.map)
        else:
            values = dict(P=m.p[i], R=m.r[i], mAP50=m.ap50[i], mAP50_95=m.ap[i])
        return {k: round(float(v), 5) for k, v in values.items()}

    out = {}
    for i, c in enumerate(results.box.ap_class_index):
        out[names[int(c)]] = {"box": block(results.box, i), "mask": block(results.seg, i)}
    out["all"] = {"box": block(results.box), "mask": block(results.seg)}
    for kind in ("box", "mask"):
        per_class = []
        for name, values in out.items():
            values[kind]["F1"] = round(f1(values[kind]["P"], values[kind]["R"]), 5)
            if name != "all":
                per_class.append(values[kind]["F1"])
        out["all"][kind]["F1"] = round(sum(per_class) / len(per_class), 5)
    return out


def table(metrics):
    """Plain-text table of a metrics_dict result."""
    header = f"{'class':<18}" + "".join(f"{k + '(' + t[0].upper() + ')':>12}" for t in ("box", "mask") for k in METRICS + ("F1",))
    rows = [header]
    for name in sorted(metrics, key=lambda n: (n != "all", n)):
        rows.append(f"{name:<18}" + "".join(f"{metrics[name][t][k]:>12.4f}" for t in ("box", "mask") for k in METRICS + ("F1",)))
    return "\n".join(rows)


def best_epoch(run_dir):
    """Epoch of best.pt, using the fitness Ultralytics applies to segmentation."""
    with open(Path(run_dir) / "results.csv", newline="") as f:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]

    def fitness(row):
        return sum(0.1 * float(row[f"metrics/mAP50({t})"]) + 0.9 * float(row[f"metrics/mAP50-95({t})"])
                   for t in ("B", "M"))

    best = max(rows, key=fitness)
    return int(float(best["epoch"])), len(rows)


def save_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

## Training

In [ ]:
def training_arguments(args):
    return dict(
        data=args.data,
        batch=args.batch,
        seed=args.seed,
        deterministic=False,
        device=args.device,
        workers=args.workers,
        project=args.project,
        name=f"seed{args.seed}",
        save_period=25,
        **SCHEDULE_VARIABILITY,
        **TRAINING,
        **ONLINE_AUGMENTATION,
    )


def train(args):
    from ultralytics import YOLO

    if args.resume:
        model = YOLO(str(Path(args.project) / f"seed{args.seed}" / "weights" / "last.pt"))
        results = model.train(resume=True)
    else:
        if not args.data:
            raise SystemExit("--data is required for a new run")
        model = YOLO(args.model)
        results = model.train(**training_arguments(args))
    run_dir = Path(model.trainer.save_dir)
    metrics = metrics_dict(results, model.names)
    epoch, epochs_run = best_epoch(run_dir)
    record = dict(seed=args.seed, model=args.model, best_epoch=epoch, epochs_run=epochs_run, validation=metrics)
    save_json(record, Path(args.project) / f"seed{args.seed}.json")
    print(table(metrics))

In [ ]:
SEED = 0  # @param {type:"integer"}
BATCH = 8  # @param {type:"integer"}

args = SimpleNamespace(
    data="/content/dataset/data.yaml", seed=SEED, resume=False, model="yolo26l-seg.pt",
    batch=BATCH, device="0", workers=11, project=f"{OUTPUT}/seeds",
)
train(args)

## Resume an interrupted run

Run the cells above again, except the training one, then this cell with the
same seed.

In [ ]:
args.resume = True
train(args)

## Summary

Mean and sample standard deviation of each metric over the available seeds.

In [ ]:
def summary(args):
    records = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(Path(args.results).glob("seed*.json"))]
    if len(records) < 2:
        raise SystemExit("at least two seed results are needed")
    print(f"seeds: {sorted(r['seed'] for r in records)}")
    print(f"{'class':<18}{'type':<6}{'metric':<10}{'mean':>9}{'sd':>9}{'min':>9}{'max':>9}")
    for name in records[0]["validation"]:
        for kind in ("box", "mask"):
            for metric in METRICS + ("F1",):
                values = [r["validation"][name][kind][metric] for r in records]
                print(f"{name:<18}{kind:<6}{metric:<10}{statistics.mean(values):>9.4f}"
                      f"{statistics.stdev(values):>9.4f}{min(values):>9.4f}{max(values):>9.4f}")

In [ ]:
summary(SimpleNamespace(results=f"{OUTPUT}/seeds"))